In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "golden")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_silver = spark.table(f"{catalogo}.{esquema_source}.products_orders_enriched")

In [0]:
df_gold_aggregated = df_silver.groupBy("department_id", "is_organic") \
    .agg(
        F.count("order_id").alias("total_orders"),
        F.sum("reordered").alias("total_reorders"),
        F.avg("add_to_cart_order").alias("avg_add_to_cart_order")
    )

In [0]:
df_gold_final = df_gold_aggregated.select(
    F.col("department_id").cast(IntegerType()),
    F.col("is_organic").cast(StringType()),
    F.col("total_orders").cast(LongType()),
    F.col("total_reorders").cast(LongType()),
    F.col("avg_add_to_cart_order").cast(DoubleType())
)

In [0]:
df_gold_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.kpis_sales_summary_partitioned")